# Stage 2: vLLM on a Colab T4 GPU

Serves the same model as the CPU stack (Qwen3-1.7B) with vLLM, then runs the same load test as
stage 7 (`docs/stage7_plan.md`, with its amendment): same prompt, 32-token cap, thinking off,
`max(40, 8 x users)` requests per level, 2 warm-ups, two rounds, same degradation rule.
Levels 32 and 64 are added (declared before running) because vLLM batches requests.
GPU memory and use are sampled with `nvidia-smi` every half second during each level.

Sources: vLLM quickstart ("Online Serving"), vLLM GPU install page (T4 is supported, compute
capability 7.5), Qwen's vLLM page (`chat_template_kwargs` to switch thinking off), the Qwen3-1.7B
model card. Runtime: Colab free tier, T4.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm==0.32.0  # pinned; the newest release at the time of writing (2026-10-18)
!vllm --version

In [ ]:
import subprocess, time, urllib.request
# Start vLLM's OpenAI-compatible server in the background; its log goes to vllm.log
log = open("vllm.log", "w")
server = subprocess.Popen(["vllm", "serve", "Qwen/Qwen3-1.7B", "--max-model-len", "4096"],
                          stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:  # wait until the server answers /v1/models, or it dies
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        break
    except OSError:
        if server.poll() is not None:
            raise SystemExit("vLLM exited, see the log:\n" + open("vllm.log").read()[-3000:])
        time.sleep(3)
print(f"vLLM ready after {time.time() - t0:.0f} s")

In [ ]:
import json
# One request with thinking off, to check the switch works and see single-user speed
body = {"model": "Qwen/Qwen3-1.7B", "messages": [{"role": "user", "content": "Say hello in five words."}],
        "chat_template_kwargs": {"enable_thinking": False}}
req = urllib.request.Request("http://localhost:8000/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json"})
t0 = time.perf_counter()
r = json.loads(urllib.request.urlopen(req, timeout=120).read())
print(repr(r["choices"][0]["message"]["content"]), r["usage"], f"{time.perf_counter() - t0:.2f} s")

In [ ]:
import concurrent.futures, math, statistics, threading
URL = "http://localhost:8000/v1/chat/completions"
PROMPT = "Count from 1 to 20, separated by spaces."  # identical to the CPU load test
MAX_TOKENS = 32
LEVELS = [1, 2, 4, 8, 16, 32, 64]  # 1..16 as on CPU, 32 and 64 added before running

def one_request():
    body = {"model": "Qwen/Qwen3-1.7B", "messages": [{"role": "user", "content": PROMPT}],
            "max_tokens": MAX_TOKENS, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}
    req = urllib.request.Request(URL, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    t0 = time.perf_counter()
    try:
        r = json.loads(urllib.request.urlopen(req, timeout=300).read())
        return 200, time.perf_counter() - t0, r["usage"]["completion_tokens"]
    except OSError as e:
        return getattr(e, "code", 0), time.perf_counter() - t0, 0

def p95(v):
    s = sorted(v); return s[max(0, math.ceil(0.95 * len(s)) - 1)]  # nearest rank, as on CPU

class GpuSampler(threading.Thread):
    # Reads GPU use (%) and memory (MiB) every 0.5 s until stopped
    def __init__(self):
        super().__init__(daemon=True); self.samples = []; self.stop = threading.Event()
    def run(self):
        while not self.stop.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                                  "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
            u, m = (float(x) for x in out.strip().split(","))
            self.samples.append((u, m)); time.sleep(0.5)

def run_level(users):
    n = max(40, 8 * users)
    for _ in range(2): one_request()  # warm-ups, not counted
    gpu = GpuSampler(); gpu.start()
    t0 = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=users) as pool:
        res = list(pool.map(lambda _: one_request(), range(n)))
    wall = time.perf_counter() - t0
    gpu.stop.set(); gpu.join()
    ok = [r for r in res if r[0] == 200]; lat = [r[1] for r in ok]
    return {"users": users, "requests": n, "ok": len(ok), "errors": n - len(ok),
            "p50_s": statistics.median(lat) if lat else None, "p95_s": p95(lat) if lat else None,
            "p95_sample_size": len(lat), "requests_per_s": len(ok) / wall,
            "tokens_per_s": sum(r[2] for r in ok) / wall, "mean_tokens": sum(r[2] for r in ok) / len(ok) if ok else None,
            "wall_s": wall, "gpu_util_mean_pct": statistics.mean(s[0] for s in gpu.samples) if gpu.samples else None,
            "gpu_mem_peak_mib": max(s[1] for s in gpu.samples) if gpu.samples else None}

def degradation_point(levels):  # identical rule to scripts/load_test.py
    for prev, cur in zip([None] + levels[:-1], levels):
        if cur["errors"] > 0:
            return cur["users"], f"{cur['errors']} of {cur['requests']} requests failed"
        if prev and prev["p95_s"] and cur["p95_s"]:
            gain = cur["requests_per_s"] / prev["requests_per_s"] - 1
            slower = cur["p95_s"] / prev["p95_s"] - 1
            if gain < 0.10 and slower > 0.50:
                return cur["users"], f"throughput {gain:+.0%}, p95 latency {slower:+.0%} vs {prev['users']} users"
    return None, "no degradation up to the highest level tested"

rounds = []
for rnd in (1, 2):
    levels = []
    for u in LEVELS:
        r = run_level(u); levels.append(r)
        print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()}), flush=True)
    point, why = degradation_point(levels)
    rounds.append({"round": rnd, "levels": levels, "degradation_point": point, "why": why})
    print(f"round {rnd}: {point} ({why})", flush=True)
points = {r["degradation_point"] for r in rounds}
verdict = (f"degrades at {points.pop()} users" if len(points) == 1 and None not in points
           else "no degradation up to 64 users" if points == {None}
           else "not stable: rounds disagree " + str([r["degradation_point"] for r in rounds]))
print("VERDICT:", verdict)

In [ ]:
# Everything needed for results/, in one block to copy out
gpu_name = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
out = {"gpu": gpu_name, "vllm": subprocess.run(["vllm", "--version"], capture_output=True, text=True).stdout.strip(),
       "model": "Qwen/Qwen3-1.7B", "prompt": PROMPT, "max_tokens": MAX_TOKENS, "rounds": rounds, "verdict": verdict}
print(json.dumps(out))